# Đánh giá test cuối — Ablation V3 full42

Attach competition `asl-signs`, Shared V3 và ba ZIP results A/B/C. B dùng `member_b_b4fd56f03a57_v3_full42_results.zip`. Bật GPU rồi **Run All**. Notebook nghiệm thu đủ 42 configs, kiểm lựa chọn `T0__HF` và hashes ba checkpoints, sau đó evaluation trên test cohort 1.141 clips.

In [ ]:
MODE = "test"  # test hoặc report
SHARED_INPUT = ""  # tự tìm Shared b4fd56f03a57
RAW_INPUT = ""  # tự tìm competition asl-signs
RESULT_INPUTS = {"A": "", "B": "", "C": ""}  # tự tìm ZIP hoặc folder results trong Kaggle Input
EXPECTED_ARCHIVE_SHA256 = {'A': 'ae638dcade7c0e4c125a3015c12e71f170fd4c8b11c71bbb38bdb6fdf400cbf9', 'B': 'ca473a5e25c6e38ffb1bf27e76859292de066ee3b40828fbbb25ec9e40d3d1dc', 'C': '9d6acb579e3f1f490dc5bb680269d45015df997ed20b24f3c230fbd48c6095a2'}
EXPECTED_WINNER = 'T0__HF'
EXPECTED_CHECKPOINTS = {'seed_42': 'f7557b21e9e4bff4fe133f277fe97c1979e32d84f6696af5dcacd625aeff8f9b', 'seed_43': 'da48fc6d8f35710e629995f1a03098f953d876b63e8cd26665475dc831fe0980', 'seed_44': '22c62c75152e2f208ee39fcd699f6b4484658ad81f300a10a025e929e36f0079'}


In [ ]:
from pathlib import Path
import hashlib, json, sys, zipfile

EXPECTED_FREEZE_ID = "b4fd56f03a57"
SEARCH_ROOT = Path("/kaggle/input")
if not SHARED_INPUT:
    candidates = list(SEARCH_ROOT.rglob("ablation_v3_shared_" + EXPECTED_FREEZE_ID + ".zip"))
    if not candidates:
        candidates = [p.parent for p in SEARCH_ROOT.rglob("freeze.json")
                      if (p.parent / "code/requirements.txt").is_file()
                      and json.loads(p.read_text()).get("freeze_id") == EXPECTED_FREEZE_ID]
    if len(candidates) != 1:
        raise ValueError(f"Tìm thấy {len(candidates)} Shared V3 inputs. Điền SHARED_INPUT tới đúng ZIP/thư mục Shared.")
    SHARED_INPUT = str(candidates[0])
source = Path(SHARED_INPUT)
if source.is_file():
    with zipfile.ZipFile(source) as archive:
        if len(archive.namelist()) != len(set(archive.namelist())) or archive.testzip():
            raise ValueError("Shared ZIP không hợp lệ.")
        payloads = {name: archive.read(name) for name in archive.namelist() if not name.endswith("/")}
else:
    payloads = {p.relative_to(source).as_posix(): p.read_bytes() for p in source.rglob("*") if p.is_file()}
frozen = json.loads(payloads["freeze.json"])
actual = {name: hashlib.sha256(payloads[name]).hexdigest() for name in frozen["files"]}
identity = hashlib.sha256(json.dumps(actual, sort_keys=True).encode()).hexdigest()[:12]
if actual != frozen["files"] or identity != EXPECTED_FREEZE_ID or frozen["freeze_id"] != identity:
    raise ValueError("Shared khác freeze của notebook này.")
manifest = json.loads(payloads["source_manifest.json"])
for name, checksum in manifest.items():
    if hashlib.sha256(payloads["code/" + name].replace(b"\r\n", b"\n")).hexdigest() != checksum:
        raise ValueError(f"Source hash lệch: {name}")
DEPLOY_ROOT = Path("/kaggle/working") / ("islr_ablation_v3_" + identity)
for name, content in payloads.items():
    target = (DEPLOY_ROOT / name).resolve()
    if not target.is_relative_to(DEPLOY_ROOT.resolve()) or "\\" in name or ":" in name:
        raise ValueError("Unsafe Shared path.")
    if target.exists() and target.read_bytes() != content:
        raise ValueError(f"Runtime source/input bị sửa: {name}")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
CODE_ROOT = DEPLOY_ROOT / "code"
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT / "reports/ablation-study-v3/tools"))
print("Đã xác minh source, config, matrix, split:", identity)


In [ ]:
import subprocess, importlib.util
required = {"IPython": "ipython", "yaml": "pyyaml", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "pandas": "pandas", "tqdm": "tqdm"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import torch
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available())
if MODE == "train" and not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Kaggle Settings trước khi training.")


In [ ]:
ORCHESTRATOR_SOURCE = '"""Explicit full-42 orchestration over unchanged frozen V3 training code."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom shutil import copytree\nimport zipfile\n\nfrom session_v3 import MemberSession, extract, relative\nimport study_v3 as study\n\n\ndef summarize_full_matrix(attempts: list[Path], output: Path) -> dict:\n    rows = [study.collect(path) for path in attempts]\n    records = {r["run_id"]: r for r in rows}\n    if len(records) != len(rows):\n        raise ValueError("Duplicate selected run in full-42 results.")\n    if rows and any(r["provenance"] != rows[0]["provenance"] or r["memberships"] != rows[0]["memberships"] for r in rows):\n        raise ValueError("Full-42 results differ in frozen source/cohort/protocol.")\n    required = [r["run_id"] for r in study.matrix()]\n    if set(records) - set(required):\n        raise ValueError("Run outside frozen 42-cell matrix.")\n    missing = [run_id for run_id in required if run_id not in records]\n    deltas = {}\n    for profile in study.PROFILES:\n        for smaller, small_regions in study.REGIONS.items():\n            for larger, large_regions in study.REGIONS.items():\n                a, b = profile + "__" + larger, profile + "__" + smaller\n                if set(small_regions) < set(large_regions) and a in records and b in records:\n                    deltas[a + " minus " + b] = study.aggregate([\n                        x - y for x, y in zip(records[a]["metrics"]["macro_f1"]["per_seed"],\n                                             records[b]["metrics"]["macro_f1"]["per_seed"])])\n    result = {"scope": "v3-full42", "freeze_id": rows[0]["provenance"]["freeze_id"] if rows else None,\n              "status": "PENDING" if missing else "COMPLETED", "completed_cells": len(rows),\n              "expected_cells": 42, "completed_fits": len(rows) * 3, "expected_fits": 126,\n              "missing": missing, "cross_plan": None,\n              "winner": study.rank(rows)[0]["run_id"] if rows and not missing else None,\n              "records": records, "paired_region_deltas": deltas}\n    study.write(output.with_suffix(".json"), result)\n    lines = ["# Nghiệm thu đủ 42 configs V3", "", f"{result[\'status\']}: {len(rows)}/42 configs.",\n             "Winner: " + (result["winner"] or "chờ đủ 42 configs"), "",\n             "| Profile | " + " | ".join(study.REGIONS) + " |", "|---|" + "---|" * len(study.REGIONS)]\n    for profile in study.PROFILES:\n        cells = []\n        for feature in study.REGIONS:\n            row = records.get(profile + "__" + feature)\n            metric = row["metrics"]["macro_f1"] if row else None\n            cells.append(f"{metric[\'mean\']:.4f} ± {metric[\'std\']:.4f}" if metric else "—")\n        lines.append("| " + profile + " | " + " | ".join(cells) + " |")\n    lines += ["", "Mở rộng phạm vi chạy thành đủ 42 configs; giữ nguyên frozen source/config/cohort.",\n              "Rule chọn winner không đổi: mean Validation Macro-F1, std thấp hơn, parameters ít hơn, run ID.",\n              "Test chỉ mở sau nghiệm thu đủ 42 configs và khóa checkpoints; std là population std của 3 seeds."]\n    output.write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    return result\n\n\nclass FullMatrixSession(MemberSession):\n    """Run every assigned YAML, reuse accepted results, and require all 42 cells."""\n\n    def queue(self) -> list[dict]:\n        return [r for r in study.matrix() if r["owner"] == self.member]\n\n    def validate_state(self) -> None:\n        super().validate_state()\n        scope_path = self.root / "FULL_MATRIX_SCOPE.json"\n        if scope_path.is_file():\n            scope = study.read(scope_path)\n            if scope.get("scope") != "v3-full42" or scope.get("freeze_id") != self.freeze_id:\n                raise ValueError("Restored full-matrix scope differs from this launcher.")\n\n    def load_cross_plan(self, source: Path) -> None:\n        raise ValueError("Full-42 launcher runs all assigned configs; CROSS_PLAN_INPUT is unnecessary.")\n\n    def restore_results(self, source: Path) -> None:\n        if self.state["selected"] or self.state["failed"] or self.state_path.exists():\n            raise FileExistsError("Restore requires an empty results session.")\n        if source.is_dir():\n            for path in source.rglob("*"):\n                if path.is_symlink():\n                    raise ValueError("Result folder contains a symlink.")\n                relative(source, path.relative_to(source).as_posix())\n            copytree(source, self.root)\n        else:\n            extract(source, self.root)\n        self.state = study.read(self.state_path)\n        self.accepted()\n        print(f"Restored {len(self.state[\'selected\'])} accepted configs; completed runs will be skipped.")\n\n    def report(self) -> dict:\n        result = super().report()\n        scope = {"scope": "v3-full42", "freeze_id": self.freeze_id,\n                 "training_source_provenance": self.provenance,\n                 "selection_scope_change": "Run all 42 existing frozen configurations instead of conditional 24/30.",\n                 "expected_cells": 42, "expected_fits": 126,\n                 "member_expected_cells": len(self.queue()),\n                 "orchestrator_sha256": study.sha(Path(__file__))}\n        study.write(self.root / "FULL_MATRIX_SCOPE.json", scope)\n        result["scope"] = scope["scope"]\n        result["remaining"] = [r["run_id"] for r in result["queue"] if r["status"] != "COMPLETED"]\n        study.write(self.root / "MEMBER_STATUS.json", result)\n        return result\n\n    def export_results(self) -> Path:\n        self.report()\n        destination = self.root.parent / f"member_{self.member.lower()}_{self.freeze_id}_v3_full42_results.zip"\n        temporary = destination.with_suffix(".zip.tmp")\n        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:\n            for path in sorted(self.root.rglob("*")):\n                name = path.relative_to(self.root)\n                if path.is_file() and name.parts[0] != "aggregate_imports" and not path.name.endswith(".tmp"):\n                    archive.write(path, name.as_posix())\n        temporary.replace(destination)\n        print(f"Results ZIP: {destination}")\n        return destination\n\n    def aggregate(self, sources: list[Path]) -> dict:\n        if self.member != "A":\n            raise ValueError("Only member A aggregates full-42 results.")\n        attempts = self.accepted()\n        members = {"A"}\n        for source in sources:\n            folder = source if source.is_dir() else self.root / "aggregate_imports" / study.sha(source)[:16]\n            if not source.is_dir() and not folder.exists():\n                extract(source, folder)\n            state = study.read(folder / "queue_status.json")\n            member = state["member"]\n            if member not in ("B", "C") or member in members or state["provenance"] != self.provenance:\n                raise ValueError("Different provenance or duplicate member.")\n            members.add(member)\n            allowed = {r["run_id"] for r in study.matrix() if r["owner"] == member}\n            for run_id, name in state["selected"].items():\n                path = relative(folder, name)\n                record = study.collect(path)\n                if run_id not in allowed or record["run_id"] != run_id:\n                    raise ValueError("Selected run differs from assigned owner/run ID.")\n                attempts.append(path)\n        result = summarize_full_matrix(attempts, self.root / "ALL_MEMBERS_FULL42_RESULTS.md")\n        self.export_results()\n        return result\n\n    def finalize_test(self, summary: dict) -> dict:\n        if (summary.get("scope") != "v3-full42" or summary.get("status") != "COMPLETED"\n                or summary.get("expected_cells") != 42 or summary.get("completed_cells") != 42\n                or set(summary.get("records", {})) != {r["run_id"] for r in study.matrix()}):\n            raise ValueError("Hold out test until the complete full-42 matrix is accepted.")\n        return super().finalize_test(summary)\n'
LAUNCHER_ROOT = Path("/kaggle/working/islr_v3_final_test_launcher")
LAUNCHER_ROOT.mkdir(parents=True, exist_ok=True)
(LAUNCHER_ROOT / "full_matrix_v3.py").write_text(ORCHESTRATOR_SOURCE, encoding="utf-8")
sys.path.insert(0, str(LAUNCHER_ROOT))
from full_matrix_v3 import FullMatrixSession
import study_v3
from src.utils.env import resolve_paths
from IPython.display import display, FileLink
import pandas as pd

if MODE not in ("test", "report"):
    raise ValueError("MODE phải là test hoặc report.")
if not RAW_INPUT:
    candidates = [p.parent for p in SEARCH_ROOT.rglob("train.csv")
                  if (p.parent / "train_landmark_files").is_dir()]
    if len(candidates) != 1:
        raise ValueError(f"Tìm thấy {len(candidates)} asl-signs roots. Điền RAW_INPUT tới đúng competition.")
    RAW_INPUT = str(candidates[0])

for member in ("A", "B", "C"):
    if not RESULT_INPUTS[member]:
        suffix = "_v3_full42_results.zip" if member == "B" else "_v3_results.zip"
        name = "member_" + member.lower() + "_" + EXPECTED_FREEZE_ID + suffix
        candidates = list(SEARCH_ROOT.rglob(name))
        if not candidates:
            candidates = []
            for path in SEARCH_ROOT.rglob("queue_status.json"):
                state = json.loads(path.read_text())
                expected = 21 if member == "B" else 7 if member == "A" else 14
                if (state.get("member") == member and state.get("freeze_id") == EXPECTED_FREEZE_ID
                        and len(state.get("selected", {})) == expected and not state.get("failed")):
                    candidates.append(path.parent)
        if len(candidates) != 1:
            raise ValueError(f"Tìm thấy {len(candidates)} results {member}. Điền RESULT_INPUTS[{member!r}] tới đúng ZIP/folder.")
        RESULT_INPUTS[member] = str(candidates[0])
    archive = Path(RESULT_INPUTS[member])
    if archive.is_file():
        with archive.open("rb") as stream:
            actual = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual != EXPECTED_ARCHIVE_SHA256[member]:
            raise ValueError(f"ZIP results {member} khác bản đã nghiệm thu.")

PERSISTENT_ROOT = Path("/kaggle/working")
root_cfg = study_v3.load_config(str(DEPLOY_ROOT / "config.yaml"))
root_cfg.env.local_root = str(PERSISTENT_ROOT)
canonical_root = resolve_paths(root_cfg.env.name, root_cfg, create_dirs=False).persistent_root
if canonical_root.resolve() != PERSISTENT_ROOT.resolve():
    raise ValueError(f"Runtime root {PERSISTENT_ROOT} differs from canonical root {canonical_root}.")
session = FullMatrixSession("A", DEPLOY_ROOT, Path(RAW_INPUT), PERSISTENT_ROOT)
if not session.state_path.exists():
    session.restore_results(Path(RESULT_INPUTS["A"]))
else:
    session.accepted()


In [ ]:
summary = session.aggregate([Path(RESULT_INPUTS["B"]), Path(RESULT_INPUTS["C"])])
if summary["status"] != "COMPLETED" or summary["completed_cells"] != 42:
    raise RuntimeError("Aggregate cần đủ 42/42 configs trước evaluation.")
if summary["winner"] != EXPECTED_WINNER:
    raise ValueError("Winner khác lựa chọn validation đã nghiệm thu.")
winner = summary["records"][EXPECTED_WINNER]
checkpoints = sorted(Path(winner["path"]).joinpath("runs").rglob("best_model.pt"))
actual = {p.parent.name: study_v3.sha(p) for p in checkpoints}
if actual != EXPECTED_CHECKPOINTS:
    raise ValueError("Checkpoints khác seeds 42/43/44 đã chọn.")
print("Đã nghiệm thu", summary["completed_cells"], "/ 42 configs; winner:", summary["winner"])
display(pd.DataFrame([{ "run_id": r["run_id"], "macro_f1": r["metrics"]["macro_f1"]["mean"] }
                      for r in study_v3.rank(list(summary["records"].values()))]).head(10))


In [ ]:
if MODE == "test":
    test_result = session.finalize_test(summary)
    print("Test set 1.141 clips, checkpoints seeds 42/43/44")
    display(pd.DataFrame(test_result["metrics"]).T)
    display(FileLink(str(session.root / "FINAL_TEST/TEST_SUMMARY.json")))
results_zip = session.export_results()
display(FileLink(str(results_zip)))


Tải Results ZIP A mới có `FINAL_TEST/TEST_SUMMARY.json`, `WINNER_LOCK.json` và ba thư mục seed chứa test reports. MODE `report` dùng để nghiệm thu và xem ranking trước evaluation.